<a href="https://colab.research.google.com/github/J-Eisenhauer/bern2/blob/main/hierarchical_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise: Hierarchical models

Author: Jakob Eisenhauer  
Date: 2026-09-20  

## Research question

Seven randomized experiments compared a control message with a descriptive social-norm message intended to encourage hotel guests to reuse their towels. The aim is to estimate the average intervention effect while accounting for differences in baseline towel-reuse rates between experiments, and to test whether the intervention increases towel reuse.

## 1. Setup

The analysis uses Bambi (Bayesian Model-Building Interface). Its a high-level layer on top of PyMC, the main Bayesian modelling library in Python.

In [1]:
!pip install bambi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.1/253.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: pytensor
    Found existing installation: pytensor 2.38.3
    Uninstalling pytensor-2.38.3:
      Successfully uninstalled pytensor-2.38.3
  Attempting uninstall: arviz
    Found existing installation: arviz 0.22.0
    Uninstalling arviz-0.22.0:
      Successfully uninstalled arviz-0.22.0
  Attempting uninstall: pymc
    Found existing inst

In [2]:
# imports and settings
import numpy as np
import pandas as pd
import bambi as bmb # bambi does the sampling.
import arviz as az #ArviZ is the toolbox for looking at the results.
import matplotlib.pyplot as plt

RANDOM_SEED = 2026


## 2. Data preparation

Each experiment contains counts for towel reuse (Yes) and non-reuse (No) in the control and social-norm groups. The following wrangling code was supplied in the exercise instructions.

In [3]:
# CODE PROVIDED BY THE INSTRUCTIONS IN THE EXERCISE

# read in data

url = 'https://raw.githubusercontent.com/luchem/bern02/main/Labs/towelData.csv'
data = pd.read_csv(url, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers or yes/no

# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 diferent studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')


In [4]:
#define an explicit 0/1 intervention indicator

combined_data['intervention'] = (combined_data['group'] == 'social').astype(int)
combined_data

,reuse,total,group,study,intervention
0,74,211,control,1,0
1,103,277,control,2,0
2,77,135,control,3,0
3,82,187,control,4,0
4,21,25,control,5,0
5,123,147,control,6,0
6,28,30,control,7,0
7,98,222,social,1,1
8,587,1318,social,2,1
9,406,655,social,3,1


## 3. Model formulation

Let $Y_{jg}$ be the number of guests who reused their towel among $n_{jg}$ guests in experiment $j$ and group $g$. Since the response is a number of successes from a known number of trials, a binomial distribution is appropriate:

$$Y_{jg} \sim Binomial(n_{jg}, p_{jg})$$

The probability is linked to a linear predictor through the logit function:

$$
logit(p_{jg}) = \ln\left(\frac{p_{jg}}{1-p_{jg}}\right) = \alpha + u_j + \beta x_g
$$

where $x_g=0$ for the control group and $x_g=1$ for the social-norm group. The experiment-specific intercepts are modelled as

$$u_j \sim Normal(0,\tau).$$

Here, $\alpha$ is the average control-group log-odds, $\beta$ is the common intervention effect, and $\tau$ describes between-study heterogeneity in log-odds. A random intercept is included because towel-reuse rates from the same experiment share a context and may have a different baseline from those in other experiments.

Bambi's built-in default priors are used.

In [5]:
# model specification using Bambi
model = bmb.Model(
    "p(reuse, total) ~ intervention + (1|study)",
    data=combined_data,
    family="binomial")
# Bambi Syntax:
#p(success, trials)
# intervention = Group
# 1|study = one random offset per study


model

       Formula: p(reuse, total) ~ intervention + (1|study)
        Family: binomial
          Link: p = logit
  Observations: 14
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 1.5)
            intervention ~ Normal(mu: 0.0, sigma: 2.0)
        
        Group-level effects
            1|study ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.6926))

## 4. Bayesian estimation

The posterior is sampled with Bambi's built-in sampler.

In [22]:
# Bayesian parameter estimation with the built-in sampler
data2 = model.fit(
    draws=2000,
    chains=4,
    cores=4, # 4 chains parallel
    target_accept=0.99,
    random_seed=RANDOM_SEED)

Output()

## 5. Parameter estimates and diagnostics

Parameter values are summarized by their posterior means and 90% credible intervals.

In [26]:
divergences = data2["sample_stats"]["diverging"].sum()
print(f'Divergent samples: {divergences.values}') # 0 divergent samples?

# posterior summaries
posterior_summary = az.summary(
    data2,
    ci_prob=0.90,
    ci_kind='hdi', # highest density interval, the narrowest interval that contains 90%
    round_to=2)

diagnostics_summary = posterior_summary[["ess_bulk", "ess_tail", "r_hat"]]
print(diagnostics_summary)


Divergent samples: 0
               ess_bulk  ess_tail  r_hat
Intercept       1760.70   1762.33    1.0
intervention    4167.85   4169.92    1.0
1|study_sigma   1515.38   2256.84    1.0
1|study[1]      1783.18   2027.37    1.0
1|study[2]      1803.80   1871.80    1.0
1|study[3]      1803.70   1908.43    1.0
1|study[4]      1738.77   1954.22    1.0
1|study[5]      2447.02   2899.36    1.0
1|study[6]      1773.77   2000.72    1.0
1|study[7]      1910.90   2335.54    1.0


All R-hat values equal 1.0 indicating that the four chains converged to the same distribution. Bulk and tail ESS exceed 1500 for every parameter, and there were no divergent transitions, so the posterior means and 90% intervals can be considered reliable.

In [18]:
mean_hdi_summary = posterior_summary[["mean", "hdi90_lb", "hdi90_ub"]]

print(mean_hdi_summary)


               mean  hdi90_lb  hdi90_ub
Intercept      0.41     -0.32      1.11
intervention   0.21      0.08      0.34
1|study_sigma  1.13      0.54      1.72
1|study[1]    -0.93     -1.64     -0.19
1|study[2]    -0.85     -1.60     -0.16
1|study[3]    -0.13     -0.81      0.63
1|study[4]    -0.62     -1.31      0.14
1|study[5]     1.15      0.24      2.05
1|study[6]     0.96      0.23      1.69
1|study[7]     0.77     -0.00      1.51


## 6. Hypothesis test

The directional hypotheses refer to the common intervention coefficient:

$$H_0: \beta \leq 0$$

$$H_1: \beta > 0.$$

The posterior probability of the null region is estimated as the proportion of posterior draws satisfying $\beta\leq0$. $H_0$ is rejected when $P(\beta\leq0\mid\text{data})<0.05$.

In [20]:
# posterior hypothesis test and effect summary
beta_chain_draw = data2.posterior['intervention'] # intervention array as (chain, draw)
beta_draws = beta_chain_draw.values.ravel() # convert to numpy array, then flatten to 1D array
prob_null = np.mean(beta_draws <= 0)

beta_mean = beta_draws.mean()
beta_hdi90 = np.array(az.hdi(beta_draws, prob=0.90))

# log odds ratio at 0 -> no effect, log odds have possible values form -inf to inf
#odds ratio at 1 -> no effect, better to interpret than log odds but not the same thing as probabilities.
odds_ratio_draws = np.exp(beta_draws)
or_mean = np.mean(odds_ratio_draws)
or_hdi90 = np.array(az.hdi(odds_ratio_draws, prob=0.90))

test_results = pd.DataFrame({
    'quantity': ['intervention coefficient beta', 'odds ratio exp(beta)'],
    'posterior_estimate': [beta_mean, or_mean],
    '90%_HDI_lower': [beta_hdi90[0], or_hdi90[0]],
    '90%_HDI_upper': [beta_hdi90[1], or_hdi90[1]]})

print(test_results.round(3))

print(f"P(beta <= 0 | data) = {prob_null:.4f}")

reject_h0 = prob_null < 0.05
print(f"Reject H0 at posterior threshold 0.05: {reject_h0}")


                        quantity  posterior_estimate  90%_HDI_lower  \
0  intervention coefficient beta               0.209          0.084   
1           odds ratio exp(beta)               1.236          1.077   

   90%_HDI_upper  
0          0.340  
1          1.393  
P(beta <= 0 | data) = 0.0039
Reject H0 at posterior threshold 0.05: True


## 7. Conclusion

The estimated common intervention coefficient was $\beta=0.209$, with a 90% HDI from 0.084 to 0.340. On the odds-ratio scale, the posterior mean was 1.236, with a 90% HDI from 1.077 to 1.393. $P(\beta\leq0\mid\mathrm{data})=0.0039$. The posterior probability of the null region is therefore below 0.05. $H_0$ is rejected in favour of a positive intervention effect.

## Reference

Scheibehenne, B., Jamil, T., & Wagenmakers, E.-J. (2016). Bayesian evidence synthesis can reconcile seemingly inconsistent results: The case of hotel towel reuse. *Psychological Science, 27*(7), 1043–1046. https://doi.org/10.1177/0956797616644081